# Fake Review Detector - Training Notebook
Dataset: Amazon fake reviews (CG = computer-generated / fake, OR = original / genuine).

Run on Google Colab (normal CPU is enough). Run all cells from top to bottom.

## 1. Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = "/content/drive/MyDrive/fake-review-detector"

import os, sys
sys.path.append(PROJECT)          # so we can import utils.py from Drive
os.makedirs(f"{PROJECT}/models", exist_ok=True)
os.makedirs(f"{PROJECT}/figures", exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import utils

print("scikit-learn version:", sklearn.__version__)

## 2. Load and clean the data

In [ ]:
df = pd.read_csv(f"{PROJECT}/data/fake reviews dataset.csv")
df = df.rename(columns={"text_": "text"})
df = df.dropna(subset=["text"]).drop_duplicates(subset=["text"])
df["label"] = df["label"].map({"CG": 1, "OR": 0})      # 1 = fake, 0 = genuine

print("Shape:", df.shape)
print(df["label"].value_counts())
df.head()

## 3. Explore the data

In [ ]:
df["word_count"] = df["text"].str.split().str.len()
df["label_name"] = df["label"].map({1: "Fake", 0: "Genuine"})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["label_name"].value_counts().plot(kind="bar", ax=axes[0], color=["tab:red", "tab:green"])
axes[0].set_title("Number of fake vs genuine reviews")
df.groupby("label_name")["word_count"].mean().plot(kind="bar", ax=axes[1], color=["tab:red", "tab:green"])
axes[1].set_title("Average number of words")
plt.tight_layout()
plt.savefig(f"{PROJECT}/figures/data_overview.png", dpi=150)
plt.show()

print(df.groupby("label_name")[["rating", "word_count"]].mean())

## 4. Make features
* **TF-IDF**: turns review text into numbers based on which words appear
* **7 handcrafted features**: word count, average word length, unique word ratio, number of '!',
  capital letters ratio, generic phrases, star rating

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
print("Train:", len(train_df), " Test:", len(test_df))

F_train = utils.features_table(train_df["text"], train_df["rating"])
F_test = utils.features_table(test_df["text"], test_df["rating"])

# Average feature values for fake vs genuine reviews
F_train.groupby(train_df["label"].map({1: "Fake", 0: "Genuine"}).values).mean()

In [ ]:
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True)
Xw_train = tfidf.fit_transform(train_df["text"].map(utils.clean_text))
Xw_test = tfidf.transform(test_df["text"].map(utils.clean_text))

scaler = StandardScaler()
X_train = hstack([Xw_train, csr_matrix(scaler.fit_transform(F_train.values))]).tocsr()
X_test = hstack([Xw_test, csr_matrix(scaler.transform(F_test.values))]).tocsr()

y_train = train_df["label"].values
y_test = test_df["label"].values

## 5. Train two models and compare
* **Model A**: only TF-IDF words
* **Model B**: TF-IDF words + our 7 features (final model)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

model_a = LogisticRegression(max_iter=2000, C=4)
model_a.fit(Xw_train, y_train)
pred_a = model_a.predict(Xw_test)

model_b = LogisticRegression(max_iter=2000, C=4)
model_b.fit(X_train, y_train)
pred_b = model_b.predict(X_test)

results = pd.DataFrame({
    "model": ["A: words only", "B: words + features"],
    "accuracy": [accuracy_score(y_test, pred_a), accuracy_score(y_test, pred_b)],
    "f1": [f1_score(y_test, pred_a), f1_score(y_test, pred_b)],
}).round(4)
results

## 6. Check the final model in detail

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

print(classification_report(y_test, pred_b, target_names=["Genuine", "Fake"]))

ConfusionMatrixDisplay.from_predictions(y_test, pred_b, display_labels=["Genuine", "Fake"], cmap="Blues")
plt.title("Confusion matrix (Model B)")
plt.savefig(f"{PROJECT}/figures/confusion_matrix.png", dpi=150)
plt.show()

## 7. Which words does the model think are fake?

In [ ]:
words = np.array(tfidf.get_feature_names_out())
weights = model_b.coef_[0][:len(words)]          # first part of the weights = words
top_fake = pd.Series(weights, index=words).sort_values().tail(15)
top_genuine = pd.Series(weights, index=words).sort_values().head(15)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
top_fake.plot(kind="barh", ax=axes[0], color="tab:red", title="Words that suggest FAKE")
top_genuine.plot(kind="barh", ax=axes[1], color="tab:green", title="Words that suggest GENUINE")
plt.tight_layout()
plt.savefig(f"{PROJECT}/figures/top_words.png", dpi=150)
plt.show()

print("Weights of our 7 features (positive = more likely fake):")
print(pd.Series(model_b.coef_[0][len(words):], index=utils.FEATURE_NAMES).round(3))

## 8. Test on some reviews from the test set (the model has never seen these)

In [ ]:
model = {"tfidf": tfidf, "scaler": scaler, "clf": model_b}

sample = test_df.sample(6, random_state=1)
probs = utils.predict_fake_probability(model, sample["text"], sample["rating"])

for text, label, p in zip(sample["text"], sample["label"], probs):
    actual = "Fake" if label == 1 else "Genuine"
    print(f"Actual: {actual:8s} | Predicted fake chance: {p:.0%} | {text[:80]}")

## 9. Save the model

In [ ]:
import joblib

joblib.dump(model, f"{PROJECT}/models/model.pkl")
results.to_csv(f"{PROJECT}/models/results.csv", index=False)
print("Saved model to", f"{PROJECT}/models/model.pkl")